# RS ai — آزمایش موتور آواتار روی GPU ابری

**برای رایانه‌های بدون GPU مناسب:** خود برنامه روی ویندوز می‌ماند و فقط مدل ویدئو در یک سرویس GPU ابری اجرا می‌شود. این دفترچه در Google Colab یا JupyterLab یک سرور GPU قابل اجراست؛ برای ایران می‌توان از سرویس‌های GPU داخلی استفاده کرد.

**هزینه:** برای مدل واقعی معمولاً GPU لازم است و رایگان بودن، میزان حافظه و سهمیه را سرویس‌دهنده تعیین می‌کند. پیش از خرید، یک تست رایگان/اعتبار آزمایشی و مشخصات GPU را بررسی کن. برای شروع، GPU انویدیا با ۲۴GB VRAM یا بیشتر هدف معقولی است، اما اجرای موفق قطعی نیست.

**حریم خصوصی:** عکس و صدا در فضای ابری منتقل و پردازش می‌شوند. فقط فایل‌هایی را آپلود کن که با این موضوع موافقی.

## اجرا
1. فایل را در Google Colab یا JupyterLab یک سرور GPU باز کن. در Colab از Runtime → Change runtime type گزینهٔ GPU را انتخاب کن.
2. سلول ۱ را اجرا کن؛ تا وقتی GPU انویدیا تأیید نشده، مدل‌ها دانلود نمی‌شوند.
3. بستهٔ ZIP خروجی RS ai را در Colab بارگذاری کن؛ یا اگر بسته نداری، عکس و فایل صوتی را جداگانه انتخاب کن. در JupyterLab، فایل‌ها را داخل پوشهٔ \`rsai_input\` قرار بده.
4. تست اول فقط ۵ ثانیه از صدا را پردازش می‌کند. پس از موفقیت می‌توانی `TEST_SECONDS = None` بگذاری تا تمام فایل صدا استفاده شود.

دفترچهٔ آزمایشی به‌صورت خودکار به پنل ویندوزی وصل نیست؛ بسته و MP4 را به‌صورت دستی بین دو محیط جابه‌جا کن.

کد مرجع: https://github.com/Omni-Avatar/OmniAvatar

In [ ]:
# سلول ۱: بررسی اینکه واقعاً GPU به محیط ابری اختصاص داده شده باشد
import subprocess, sys, shutil, pathlib
r = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(r.stdout if r.returncode == 0 else r.stderr)
if r.returncode != 0:
    raise RuntimeError('GPU در این نشست در دسترس نیست. Runtime را روی GPU بگذارید و دوباره اجرا کنید؛ مدل‌ها را هنوز دانلود نکنید.')

# Check hardware and free disk before downloading multi-GB weights.
gpu_mem = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
    text=True, capture_output=True
)
if gpu_mem.returncode != 0 or not gpu_mem.stdout.strip():
    raise RuntimeError('خواندن حافظهٔ GPU ممکن نشد؛ پیش از دانلود مدل‌ها مشکل GPU را بررسی کن.')
gpu_mib = int(gpu_mem.stdout.strip().splitlines()[0])
disk_root = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
free_gb = shutil.disk_usage(disk_root).free / (1024 ** 3)
print(f'GPU memory: {gpu_mib / 1024:.1f} GB | Free disk: {free_gb:.1f} GB')
if gpu_mib < 12000:
    raise RuntimeError('این GPU کمتر از 12GB حافظه دارد؛ برای جلوگیری از دانلود بی‌فایدهٔ چندین گیگابایت مدل، اجرا متوقف شد.')
if free_gb < 25:
    raise RuntimeError('کمتر از 25GB فضای خالی وجود دارد؛ پیش از دانلود وزن‌ها فضای بیشتری آزاد کن.')

In [ ]:
# سلول ۲: دریافت سورس و وابستگی‌های رسمی OmniAvatar
# دانلودها در فضای موقت Colab انجام می‌شوند، نه روی رایانهٔ ویندوزی شما.
import os, pathlib, subprocess, textwrap
WORK = pathlib.Path('/content/RS-ai-cloud')
REPO = pathlib.Path('/content/OmniAvatar')
WORK.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/Omni-Avatar/OmniAvatar',str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip','setuptools','wheel'], check=True)
# Official repo pins this CUDA-enabled PyTorch family. If Colab's runtime is incompatible,
# stop here and keep the full error; do not repeatedly reinstall packages blindly.
subprocess.run([sys.executable,'-m','pip','install','torch==2.4.0','torchvision==0.19.0','torchaudio==2.4.0','--index-url','https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt','huggingface_hub[cli]'], check=True)
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch به GPU دسترسی ندارد؛ قبل از دانلود مدل‌ها مشکل محیط را برطرف کنید.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# سلول ۳: دانلود وزن‌های موردنیاز؛ ممکن است چندین گیگابایت باشد
# این مرحله را فقط در محیط ابری دارای GPU اجرا کنید.
env = os.environ.copy()
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
model_dir = REPO / 'pretrained_models'
model_dir.mkdir(exist_ok=True)
downloads = [
    ['hf','download','Wan-AI/Wan2.1-T2V-1.3B','--local-dir',str(model_dir/'Wan2.1-T2V-1.3B')],
    ['hf','download','facebook/wav2vec2-base-960h','--local-dir',str(model_dir/'wav2vec2-base-960h')],
    ['hf','download','OmniAvatar/OmniAvatar-1.3B','--local-dir',str(model_dir/'OmniAvatar-1.3B')],
]
for cmd in downloads:
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
print('\nDownload step completed. Disk usage:')
subprocess.run(['du','-sh',str(model_dir)], check=False)

# Stop before inference if a large model download was incomplete.
expected = [
    model_dir / 'Wan2.1-T2V-1.3B' / 'diffusion_pytorch_model.safetensors',
    model_dir / 'Wan2.1-T2V-1.3B' / 'models_t5_umt5-xxl-enc-bf16.pth',
    model_dir / 'Wan2.1-T2V-1.3B' / 'Wan2.1_VAE.pth',
    model_dir / 'OmniAvatar-1.3B' / 'pytorch_model.pt',
    model_dir / 'wav2vec2-base-960h' / 'config.json',
]
missing = [str(item) for item in expected if not item.is_file()]
if missing:
    raise FileNotFoundError('Model files missing after download; retry the download cell before inference: ' + ', '.join(missing))

In [ ]:
# سلول ۴: دریافت بستهٔ RS ai یا انتخاب عکس و صدا
from IPython.display import Video, display, FileLink
import shutil, re, zipfile

image_exts = {'.png', '.jpg', '.jpeg', '.webp'}
audio_exts = {'.wav', '.mp3', '.m4a', '.aac', '.flac', '.ogg'}
try:
    from google.colab import files as colab_files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    uploaded = colab_files.upload()
    input_files = [pathlib.Path.cwd() / n for n in uploaded]
else:
    INPUT_DIR = pathlib.Path.cwd() / 'rsai_input'
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('بستهٔ ZIP یا عکس و فایل صوتی را داخل این پوشه قرار بده، سپس سلول را دوباره اجرا کن:')
    print(INPUT_DIR)
    input_files = [p for p in INPUT_DIR.iterdir() if p.is_file()]

zip_files = sorted([p for p in input_files if p.suffix.lower() == '.zip'])
bundle_dir = WORK / 'uploaded_bundle'
prompt_file = None
if zip_files:
    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_files[0]) as archive:
        root = bundle_dir.resolve()
        for member in archive.infolist():
            target = (bundle_dir / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError('مسیر نامعتبر داخل بستهٔ ZIP؛ استخراج متوقف شد.')
        archive.extractall(bundle_dir)
    input_files = [p for p in bundle_dir.rglob('*') if p.is_file()]
    prompt_file = bundle_dir / 'prompt.txt'

image_files = [p for p in input_files if p.suffix.lower() in image_exts]
audio_files = [p for p in input_files if p.suffix.lower() in audio_exts]
if not image_files or not audio_files:
    raise ValueError('یک بستهٔ RS ai معتبر یا یک عکس JPG/PNG به‌همراه فایل صوتی WAV/MP3 لازم است.')

image_in = next((p for p in image_files if p.name.lower() == 'reference.png'), sorted(image_files)[0])
audio_in = next((p for p in audio_files if p.name.lower() == 'speech_16k.wav'), sorted(audio_files)[0])
job = WORK / 'input'
job.mkdir(parents=True, exist_ok=True)
image_path = job / ('reference' + image_in.suffix.lower())
shutil.copy2(image_in, image_path)
audio_path = job / 'speech_16k.wav'

TEST_SECONDS = 5  # پس از موفقیت تست کوتاه، این مقدار را None کن و سلول‌های ۴ تا ۶ را دوباره اجرا کن.
ffmpeg_cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', str(audio_in), '-vn', '-ac', '1', '-ar', '16000']
if TEST_SECONDS is not None:
    ffmpeg_cmd += ['-t', str(TEST_SECONDS)]
ffmpeg_cmd += [str(audio_path)]
subprocess.run(ffmpeg_cmd, check=True)

if prompt_file and prompt_file.is_file():
    PROMPT = prompt_file.read_text(encoding='utf-8').strip()
else:
    PROMPT = ('Photorealistic adult presenter speaking naturally to camera; preserve the identity and appearance of the reference person; natural head movement and expressive but plausible hand and arm gestures; stable camera; coherent body motion; no text, subtitles or logos.')
PROMPT = PROMPT.replace('@@', ' ').replace('\n', ' ')
input_file = WORK / 'rsai_input.txt'
input_file.write_text(f'{PROMPT}@@{image_path}@@{audio_path}\n', encoding='utf-8')
print('Reference image:', image_path.name)
print('Audio duration mode:', f'{TEST_SECONDS} seconds test' if TEST_SECONDS else 'full audio')
print('Input file prepared:', input_file)


In [ ]:
# سلول ۵: اجرای آزمایشی مدل. تنظیمات کم‌حافظه و تعداد گام پایین برای اولین تست.
# این تنظیمات ممکن است کیفیت پایین‌تری بدهند؛ برای کیفیت بیشتر NUM_STEPS را به 20 یا بالاتر افزایش دهید.
# Short smoke test is intentionally fast/low quality. Full run uses the recommended 25 steps.
NUM_STEPS = 8 if TEST_SECONDS is not None else 25
MAX_TOKENS = 15000
print('Inference quality mode:', 'short smoke test' if TEST_SECONDS is not None else 'full run (25 steps)')
HYPARAMS = f'dtype=fp16,num_steps={NUM_STEPS},max_tokens={MAX_TOKENS},num_persistent_param_in_dit=0,tea_cache_l1_thresh=0.14'
cmd = ['torchrun','--standalone','--nproc_per_node=1','scripts/inference.py','--config','configs/inference_1.3B.yaml','--input_file',str(input_file),'--hp='+HYPARAMS]
print('Running inference. If it reports CUDA out-of-memory, this free GPU is not sufficient for this configuration.')
print('Command:', ' '.join(cmd))
run = subprocess.run(cmd, cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_path = WORK / 'omniavatar.log'
log_path.write_text(run.stdout or '', encoding='utf-8', errors='replace')
print((run.stdout or '')[-12000:])
if run.returncode != 0:
    raise RuntimeError(f'OmniAvatar failed with code {run.returncode}. Full log: {log_path}')
outputs = sorted((REPO/'demo_out').rglob('result_000.mp4'), key=lambda p:p.stat().st_mtime, reverse=True)
if not outputs:
    raise FileNotFoundError('Inference returned successfully but result_000.mp4 was not found. Inspect the log.')
video_path = outputs[0]
print('Generated video:', video_path, f'({video_path.stat().st_size:,} bytes)')
display(Video(str(video_path), embed=True))

In [ ]:
# سلول ۶: ذخیره و دریافت MP4
download_path = WORK / 'RS-ai-avatar-cloud.mp4'
shutil.copy2(video_path, download_path)
if IS_COLAB:
    colab_files.download(str(download_path))
else:
    print('از پیوند زیر فایل ویدئو را دانلود کن:')
    display(FileLink(str(download_path)))